In [1]:
# Install required libraries (runs in Colab)
!pip install -q sentence-transformers faiss-cpu openai pandas tqdm || true



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
import pickle


c:\Users\Kulsoom\nafa-ai-pipline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
import pandas as pd

csv_path = "PSX_Combined_100_plus.csv"

try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    raise Exception(f"Could not find file at {csv_path}. Please update the path.")
except Exception as e:
    raise e

print("Loaded rows:", len(df))
df.head(5)

Loaded rows: 176


,Ticker,CompanyName,Status,OpenPrice,ClosePrice,DailyReturn%,SharesOutstanding,MarketCap,CAGR,Volatility,RiskScore,RiskLevel,Sector
0,OGDC,Oil & Gas Development Company Limited,Halal (Shariah Compliant),761.59,739.99,-2.8362,822172875,608399705771,-0.0384,2.7973,0.2989,Low,Oil & Gas
1,UBL,United Bank Limited,Not Halal (Conventional),680.74,692.16,1.6776,329237316,227884900643,0.0944,0.3230,0.0000,Low,Banking
2,MARI,Mari Petroleum Company Limited,Halal (Shariah Compliant),440.43,436.51,-0.8900,1555999102,679209168014,0.0550,0.8357,0.0561,Low,Oil & Gas
3,MEBL,Meezan Bank Limited,Halal (Shariah Compliant),65.66,61.18,-6.8230,280995032,17191276058,0.1448,3.5086,0.2785,Low,Banking
4,FFC,Fauji Fertilizer Company Limited,Halal (Shariah Compliant),923.02,910.20,-1.3889,2595460470,2362388119794,0.0685,3.4943,0.3152,Moderate,Fertilizer


In [15]:
required_cols = ['Ticker'	,'Status',	'OpenPrice',	'ClosePrice',	'DailyReturn%',	'SharesOutstanding',	'MarketCap',	'CAGR',	'Volatility',	'RiskScore',	'RiskLevel','Sector']
missing = [c for c in required_cols if c not in df.columns]
print('Missing columns from CSV (if any):', missing)
def row_to_doc(row):
    parts = [
        f"Ticker: {row.get('Ticker','')}.",
        f"Company Name: {row.get('CompanyName','')}.",
        f"Status: {row.get('Status','')}.",
        f"Open Price: {row.get('OpenPrice','')}.",
        f"Close Price: {row.get('ClosePrice','')}.",
        f"Daily Return Percentage: {row.get('DailyReturn%','')}.",
        f"Shares Outstanding: {row.get('SharesOutstanding','')}.",
        f"Market Cap: {row.get('MarketCap','')}.",
        f"CAGR: {row.get('CAGR','')}.",
        f"Volatility: {row.get('Volatility','')}.",
        f"Risk Score: {row.get('RiskScore','')}.",
        f"Risk Level: {row.get('RiskLevel','')}.",
        f"Sector: {row.get('Sector','')}.", 
    ]
    return ' '.join(parts)

df['document'] = df.apply(row_to_doc, axis=1)
print('Sample document:')
print(df['document'].iloc[0])

Missing columns from CSV (if any): []
Sample document:
Ticker: OGDC. Company Name: Oil & Gas Development Company Limited. Status: Halal (Shariah Compliant). Open Price: 761.59. Close Price: 739.99. Daily Return Percentage: -2.8362. Shares Outstanding: 822172875. Market Cap: 608399705771. CAGR: -0.0384. Volatility: 2.7973. Risk Score: 0.2989. Risk Level: Low. Sector: Oil & Gas.


In [16]:
model_name = 'all-MiniLM-L6-v2'
print('Loading model:', model_name)
model = SentenceTransformer(model_name)

docs = df['document'].tolist()
batch_size = 64
embeddings = model.encode(docs, show_progress_bar=True, batch_size=batch_size)
embeddings = np.array(embeddings).astype('float32')
print('Embeddings shape:', embeddings.shape)

Loading model: all-MiniLM-L6-v2


Batches: 100%|██████████| 3/3 [00:05<00:00,  1.69s/it]

Embeddings shape: (176, 384)


In [ ]:
model_name = 'all-MiniLM-L6-v2'
print('Loading model:', model_name)
model = SentenceTransformer(model_name)

docs = df['document'].tolist()
batch_size = 64
embeddings = model.encode(docs, show_progress_bar=True, batch_size=batch_size)
embeddings = np.array(embeddings).astype('float32')
print('Embeddings shape:', embeddings.shape)


Loading model: all-MiniLM-L6-v2


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.46s/it]

Embeddings shape: (103, 384)


In [18]:
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d) 
faiss.normalize_L2(embeddings)
index.add(embeddings)
print('FAISS index contains', index.ntotal, 'vectors')

faiss.write_index(index, 'faiss_index_file_file.idx')
with open('faiss_metadata_file_file.pkl', 'wb') as f:
    pickle.dump(df.to_dict(orient='records'), f)

print('Saved faiss_index_file.idx and faiss_metadata_file.pkl to current directory. You can download them using Colab file browser or files.download().')

FAISS index contains 176 vectors
Saved faiss_index_file.idx and faiss_metadata_file.pkl to current directory. You can download them using Colab file browser or files.download().


In [ ]:
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

import os
import numpy as np
import faiss
import pandas as pd

gemini_api_key = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=gemini_api_key)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

def gemini_refine_query(raw_query):
    """
    Uses Gemini to reformulate a vague user query into a clear financial query.
    """
    prompt = f"""
    You are a financial assistant. Reformulate this user query into a clear, factual financial search query.
    Example:
    Input: "I am low risk person"
    Output: "low risk halal and nonhalal investment companies in Pakistan"
    
    Now reformulate:
    {raw_query}
    """
    response = gemini_model.generate_content(prompt)
    refined_query = response.text.strip()
    print(f"Refined query: {refined_query}")
    return refined_query

def embed_query_with_sentence_transformer(query):
    refined_query = gemini_refine_query(query)
    vec = sentence_model.encode([refined_query], convert_to_numpy=True)
    vec = np.array(vec).astype("float32")
    faiss.normalize_L2(vec)
    return vec

def search(query, df, index, top_k=170, risk_filter=None, halal_filter=None, include_all_halal=False):
    """
    Uses Gemini to refine the query, then embeds it with SentenceTransformer,
    performs FAISS search, and filters results.
    
    include_all_halal=True -> includes both Halal and Non-Halal options if risk_filter is provided.
    """
    qvec = embed_query_with_sentence_transformer(query)
    D, I = index.search(qvec, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = df.iloc[idx].to_dict()
        meta["_score"] = float(score)

        if risk_filter and str(meta.get("RiskLevel", "")).lower() != str(risk_filter).lower():
            continue

        if not include_all_halal:
            if halal_filter and str(meta.get("HalalStatus", "")).lower() != str(halal_filter).lower():
                continue

        results.append(meta)
    return results


query = "" \
" companies with stable returns"
hits = search(
    query,
    df,
    index,
    top_k=170,           
    risk_filter="Low",  
    include_all_halal=True  
)

print(f"Found {len(hits)} results:")
print(f"Found {len(hits)} results:")
for h in hits:
    print(h['Ticker'], h['RiskLevel'], h['Status'], h['_score']
)

Refined query: low volatility stocks
Found 28 results:
Found 28 results:
HBLT Low Not Halal (Conventional) 0.3155214190483093
HINO Low Halal (Shariah Compliant) 0.30942028760910034
LSE Low Halal (Shariah Compliant) 0.30939263105392456
KEL Low Halal (Shariah Compliant) 0.3089461028575897
ELSM Low Halal (Shariah Compliant) 0.30658379197120667
PREMA Low Halal (Shariah Compliant) 0.3009646534919739
MARI2 Low Halal (Shariah Compliant) 0.29532480239868164
SPL Low Halal (Shariah Compliant) 0.29406872391700745
ENGROH Low Halal (Shariah Compliant) 0.291415810585022
EFUG Low Not Halal (Conventional) 0.28938156366348267
DOL Low Halal (Shariah Compliant) 0.28900158405303955
ENGP Low Halal (Shariah Compliant) 0.2887181043624878
JUBS Low Halal (Shariah Compliant) 0.28804516792297363
GSPM Low Halal (Shariah Compliant) 0.2859243154525757
JATM Low Halal (Shariah Compliant) 0.28368934988975525
ENGRO Low Halal (Shariah Compliant) 0.2788344621658325
MTL2 Low Halal (Shariah Compliant) 0.2779524326324463
EF

In [21]:
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d) 
faiss.normalize_L2(embeddings)
index.add(embeddings)
print('FAISS index contains', index.ntotal, 'vectors')

faiss.write_index(index, 'faiss_index_file.idx')
with open('faiss_metadata_file.pkl', 'wb') as f:
    pickle.dump(df.to_dict(orient='records'), f)



FAISS index contains 176 vectors


In [ ]:

df.to_csv('psx_processed_for_rag.csv', index=False)

Saved psx_processed_for_rag.csv, faiss_index.idx, faiss_metadata.pkl
